# Trees — LeetCode Questions (12 Qs)

Key traversals:
- DFS (depth-first): preorder, inorder, postorder — use recursion or stack
- BFS (breadth-first): level order — use a queue

1.  104 – Maximum Depth of Binary Tree
2.  226 – Invert Binary Tree
3.  100 – Same Tree
4.  102 – Binary Tree Level Order Traversal
5.  543 – Diameter of Binary Tree
6.  110 – Balanced Binary Tree
7.  112 – Path Sum
8.  199 – Binary Tree Right Side View
9.   98 – Validate Binary Search Tree
10. 235 – Lowest Common Ancestor of BST
11. 230 – Kth Smallest Element in BST
12. 105 – Construct BT from Preorder & Inorder

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build(vals):
    """Build tree from level-order list, None = missing node."""
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


# 1. LeetCode 104 – Maximum Depth of Binary Tree

Return the maximum depth (number of nodes along longest root-to-leaf path).

Input: [3,9,20,null,null,15,7]  Output: 3

## Dry Run (DFS Recursion)

```
        3          depth=1
       / \
      9   20       depth=2
         / \
        15   7     depth=3
```

| call | node | left depth | right depth | return |
|------|------|-----------|-------------|--------|
| maxDepth(3) | 3 | maxDepth(9) | maxDepth(20) | 1+max(1,2)=3 |
| maxDepth(9) | 9 | maxDepth(None)=0 | maxDepth(None)=0 | 1+0=1 |
| maxDepth(20) | 20 | maxDepth(15)=1 | maxDepth(7)=1 | 1+max(1,1)=2 |

In [ ]:
class Solution:
    def maxDepth(self, root: Optional[TreeNode]) -> int:
        if root is None:
            return 0
        left_depth  = self.maxDepth(root.left)
        right_depth = self.maxDepth(root.right)
        return 1 + max(left_depth, right_depth)

root = build([3, 9, 20, None, None, 15, 7])
print(Solution().maxDepth(root))  # 3

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – visit every node once | O(h) – call stack depth = tree height |
| h = O(log n) balanced, O(n) skewed | **Final: O(h)** |
| **Final: O(n)** | |

# 2. LeetCode 226 – Invert Binary Tree

Swap left and right children at every node recursively.

Input: [4,2,7,1,3,6,9]  Output: [4,7,2,9,6,3,1]

## Dry Run

```
Before:        After:
    4              4
   / \            / \
  2   7    →    7   2
 /\ /\         /\ /\
1  3 6  9      9  6 3  1
```

| call | action |
|------|--------|
| invert(4) | swap children: left=7, right=2, recurse both |
| invert(7) | swap 6,9 → left=9, right=6 |
| invert(2) | swap 1,3 → left=3, right=1 |
| leaves | None → return None |

In [ ]:
class Solution:
    def invertTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left, root.right = root.right, root.left   # swap
        self.invertTree(root.left)
        self.invertTree(root.right)
        return root

root = build([4, 2, 7, 1, 3, 6, 9])
Solution().invertTree(root)
print(root.left.val, root.right.val)  # 7, 2

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – visit every node | O(h) – recursion stack |
| **Final: O(n)** | **Final: O(h)** |

# 3. LeetCode 100 – Same Tree

Two trees are the same if structurally identical and nodes have same values.

Input: p=[1,2,3], q=[1,2,3]  Output: True

## Dry Run

| step | p.val | q.val | match? | recurse |
|------|-------|-------|--------|---------|
| isSameTree(1,1) | 1 | 1 | yes | check left and right |
| isSameTree(2,2) | 2 | 2 | yes | check leaves |
| isSameTree(3,3) | 3 | 3 | yes | check leaves |
| all leaves None | None | None | True | done |
| Result: True | | | | |

In [ ]:
class Solution:
    def isSameTree(self, p: Optional[TreeNode], q: Optional[TreeNode]) -> bool:
        if p is None and q is None:
            return True
        if p is None or q is None:       # one is None, other isn't
            return False
        if p.val != q.val:
            return False
        return self.isSameTree(p.left, q.left) and self.isSameTree(p.right, q.right)

p = build([1, 2, 3])
q = build([1, 2, 3])
print(Solution().isSameTree(p, q))   # True
p2 = build([1, 2])
q2 = build([1, None, 2])
print(Solution().isSameTree(p2, q2)) # False

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – compare every node | O(h) – recursion stack |
| **Final: O(n)** | **Final: O(h)** |

# 4. LeetCode 102 – Binary Tree Level Order Traversal

Return nodes level by level (BFS with a queue).

Input: [3,9,20,null,null,15,7]  Output: [[3],[9,20],[15,7]]

## Dry Run (BFS with queue)

| Level | Queue before | Processing | Queue after | level result |
|-------|-------------|------------|-------------|-------------|
| 0 | [3] | pop 3, add children 9,20 | [9,20] | [3] |
| 1 | [9,20] | pop 9(no child), pop 20 add 15,7 | [15,7] | [9,20] |
| 2 | [15,7] | pop 15(no child), pop 7(no child) | [] | [15,7] |
| done | [] | | | [[3],[9,20],[15,7]] |

In [ ]:
class Solution:
    def levelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if not root:
            return []
        result = []
        queue = deque([root])
        while queue:
            level_size = len(queue)
            level = []
            for _ in range(level_size):      # process entire current level
                node = queue.popleft()
                level.append(node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            result.append(level)
        return result

root = build([3, 9, 20, None, None, 15, 7])
print(Solution().levelOrder(root))  # [[3],[9,20],[15,7]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – every node enqueued/dequeued once | O(n) – queue holds at most one full level |
| **Final: O(n)** | **Final: O(n)** |

# 5. LeetCode 543 – Diameter of Binary Tree

Diameter = longest path between any two nodes (may not pass through root).
Path length = number of edges.

Input: [1,2,3,4,5]  Output: 3  (path 4→2→1→3 or 5→2→1→3)

## Dry Run

```
    1
   / \
  2   3
 / \
4   5
```

At each node, diameter through it = left_depth + right_depth

| node | left_h | right_h | diameter here | global max |
|------|--------|---------|---------------|------------|
| 4 | 0 | 0 | 0 | 0 |
| 5 | 0 | 0 | 0 | 0 |
| 2 | 1 | 1 | 2 | 2 |
| 3 | 0 | 0 | 0 | 2 |
| 1 | 2 | 1 | 3 | **3** |

In [ ]:
class Solution:
    def diameterOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        self.max_diam = 0

        def depth(node):
            if node is None:
                return 0
            left  = depth(node.left)
            right = depth(node.right)
            self.max_diam = max(self.max_diam, left + right)
            return 1 + max(left, right)

        depth(root)
        return self.max_diam

root = build([1, 2, 3, 4, 5])
print(Solution().diameterOfBinaryTree(root))  # 3

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – single DFS traversal | O(h) – recursion stack |
| **Final: O(n)** | **Final: O(h)** |

# 6. LeetCode 110 – Balanced Binary Tree

A tree is balanced if every node's left-right subtree heights differ by at most 1.

Input: [3,9,20,null,null,15,7]  Output: True

## Dry Run

Check height bottom-up; return -1 if any subtree is unbalanced.

| node | left_h | right_h | |diff| | balanced? | return |
|------|--------|---------|------|-----------|--------|
| 9 | 0 | 0 | 0 | yes | 1 |
| 15 | 0 | 0 | 0 | yes | 1 |
| 7 | 0 | 0 | 0 | yes | 1 |
| 20 | 1 | 1 | 0 | yes | 2 |
| 3 | 1 | 2 | 1 | yes | 3 |
| Result: True | | | | | |

In [ ]:
class Solution:
    def isBalanced(self, root: Optional[TreeNode]) -> bool:
        def check_height(node):
            if node is None:
                return 0
            left  = check_height(node.left)
            right = check_height(node.right)
            if left == -1 or right == -1:
                return -1                       # propagate unbalanced signal
            if abs(left - right) > 1:
                return -1                       # this subtree is unbalanced
            return 1 + max(left, right)

        return check_height(root) != -1

root = build([3, 9, 20, None, None, 15, 7])
print(Solution().isBalanced(root))  # True
root2 = build([1, 2, 2, 3, 3, None, None, 4, 4])
print(Solution().isBalanced(root2)) # False

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – each node visited once | O(h) – recursion stack |
| **Final: O(n)** | **Final: O(h)** |

> Naive O(n^2): compute height separately for each node
> Optimized O(n): combine height+balance check in single DFS

# 7. LeetCode 112 – Path Sum

Return True if tree has root-to-leaf path where sum of node values = targetSum.

Input: root=[5,4,8,11,null,13,4,7,2], targetSum=22  Output: True  (5→4→11→2)

## Dry Run (subtract target at each level)

| call | node.val | remaining | at leaf? | result |
|------|----------|-----------|----------|--------|
| pathSum(5, 22) | 5 | 22-5=17 | no | recurse left+right |
| pathSum(4, 17) | 4 | 17-4=13 | no | recurse left |
| pathSum(11, 13) | 11 | 13-11=2 | no | recurse left+right |
| pathSum(7, 2) | 7 | 2-7=-5 | yes | -5!=0 → False |
| pathSum(2, 2) | 2 | 2-2=0 | yes | 0==0 → **True** |

In [ ]:
class Solution:
    def hasPathSum(self, root: Optional[TreeNode], targetSum: int) -> bool:
        if root is None:
            return False
        if root.left is None and root.right is None:    # leaf node
            return root.val == targetSum
        remaining = targetSum - root.val
        return self.hasPathSum(root.left, remaining) or self.hasPathSum(root.right, remaining)

root = build([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print(Solution().hasPathSum(root, 22))   # True
print(Solution().hasPathSum(root, 27))   # False

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – visit every node in worst case | O(h) – recursion stack |
| **Final: O(n)** | **Final: O(h)** |

# 8. LeetCode 199 – Binary Tree Right Side View

Return values visible when looking at tree from the right side.
= last node of each level in BFS.

Input: [1,2,3,null,5,null,4]  Output: [1,3,4]

## Dry Run (BFS, take last element of each level)

| Level | Nodes in level | Last (visible) |
|-------|---------------|----------------|
| 0 | [1] | 1 |
| 1 | [2, 3] | 3 |
| 2 | [5, 4] | 4 |
| Result: [1, 3, 4] | | |

In [ ]:
class Solution:
    def rightSideView(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        result = []
        queue = deque([root])
        while queue:
            level_size = len(queue)
            for i in range(level_size):
                node = queue.popleft()
                if i == level_size - 1:        # last node in level = rightmost
                    result.append(node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
        return result

root = build([1, 2, 3, None, 5, None, 4])
print(Solution().rightSideView(root))  # [1, 3, 4]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – BFS visits every node | O(n) – queue holds widest level |
| **Final: O(n)** | **Final: O(n)** |

# 9. LeetCode 98 – Validate Binary Search Tree

BST property: left subtree values < node < right subtree values (all levels).
Pass a valid range [min_val, max_val] down recursively.

Input: [2,1,3]  Output: True
Input: [5,1,4,null,null,3,6]  Output: False (4 is in right subtree of 5, but 3<5)

## Dry Run

Input: [5,1,4,null,null,3,6]

| call | node | min_limit | max_limit | node.val in range? | valid? |
|------|------|-----------|-----------|-------------------|--------|
| validate(5, -inf, +inf) | 5 | -inf | +inf | yes | recurse |
| validate(1, -inf, 5) | 1 | -inf | 5 | yes | both children None → True |
| validate(4, 5, +inf) | 4 | 5 | +inf | 4 < 5 → **False** | stop |
| Result: False | | | | | |

In [ ]:
class Solution:
    def isValidBST(self, root: Optional[TreeNode]) -> bool:
        def validate(node, min_val, max_val):
            if node is None:
                return True
            if not (min_val < node.val < max_val):
                return False
            return (validate(node.left,  min_val,   node.val) and
                    validate(node.right, node.val,  max_val))

        return validate(root, float('-inf'), float('inf'))

root1 = build([2, 1, 3])
print(Solution().isValidBST(root1))   # True
root2 = build([5, 1, 4, None, None, 3, 6])
print(Solution().isValidBST(root2))   # False

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – every node visited once | O(h) – recursion stack |
| **Final: O(n)** | **Final: O(h)** |

# 10. LeetCode 235 – Lowest Common Ancestor of BST

LCA = deepest node that has both p and q as descendants (or is p or q itself).
Use BST property: if both p,q < node → go left; both > node → go right; else current is LCA.

Input: root=[6,2,8,0,4,7,9,null,null,3,5], p=2, q=8  Output: 6

## Dry Run

| node | p=2 | q=8 | both left? | both right? | action |
|------|-----|-----|-----------|------------|--------|
| 6 | 2<6 | 8>6 | no | no | **LCA = 6** |

Another case: p=2, q=4
| node | p=2 | q=4 | action |
|------|-----|-----|--------|
| 6 | both < 6 | | go left |
| 2 | p=2, q=4>2 | | **LCA = 2** |

In [ ]:
class Solution:
    def lowestCommonAncestor(self, root: TreeNode, p: TreeNode, q: TreeNode) -> TreeNode:
        while root:
            if p.val < root.val and q.val < root.val:
                root = root.left          # both in left subtree
            elif p.val > root.val and q.val > root.val:
                root = root.right         # both in right subtree
            else:
                return root               # split point = LCA

root = build([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
p = TreeNode(2)
q = TreeNode(8)
lca = Solution().lowestCommonAncestor(root, p, q)
print(lca.val)  # 6

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(h) – only traverse one path | O(1) – iterative, no extra space |
| h = O(log n) balanced, O(n) skewed | **Final: O(1)** |
| **Final: O(h)** | |

# 11. LeetCode 230 – Kth Smallest Element in BST

Inorder traversal of BST gives sorted order. Stop at kth element.

Input: root=[3,1,4,null,2], k=1  Output: 1

## Dry Run (Inorder: Left → Node → Right)

Tree: [3,1,4,null,2]

| visit order | node.val | k countdown | action |
|-------------|----------|-------------|--------|
| 1st | 1 | k=1 → 1==1 | return 1 |

Inorder visits: 1, 2, 3, 4 (sorted)
k=1 → 1st smallest = 1

In [ ]:
class Solution:
    def kthSmallest(self, root: Optional[TreeNode], k: int) -> int:
        self.k = k
        self.result = None

        def inorder(node):
            if node is None or self.result is not None:
                return
            inorder(node.left)
            self.k -= 1
            if self.k == 0:
                self.result = node.val
                return
            inorder(node.right)

        inorder(root)
        return self.result

root = build([3, 1, 4, None, 2])
print(Solution().kthSmallest(root, 1))  # 1
print(Solution().kthSmallest(root, 3))  # 3

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(h + k) – go left until bottom, then k steps | O(h) – recursion stack |
| **Final: O(h + k)** | **Final: O(h)** |

# 12. LeetCode 105 – Construct Binary Tree from Preorder and Inorder

Preorder: [root, left, right] — first element is always root
Inorder: [left, root, right] — root splits left and right subtrees

Input: preorder=[3,9,20,15,7], inorder=[9,3,15,20,7]  Output: [3,9,20,null,null,15,7]

## Dry Run

preorder=[3,9,20,15,7], inorder=[9,3,15,20,7]

| step | preorder[0] | inorder split | left inorder | right inorder |
|------|------------|---------------|-------------|--------------|
| 1 | 3 (root) | idx=1 in inorder | [9] | [15,20,7] |
| 2 | 9 (left root) | idx=0 | [] | [] → leaf |
| 3 | 20 (right root) | idx=1 in [15,20,7] | [15] | [7] |
| 4 | 15 (left of 20) | leaf | | |
| 5 | 7 (right of 20) | leaf | | |

In [ ]:
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        if not preorder or not inorder:
            return None
        root_val = preorder[0]
        root = TreeNode(root_val)
        mid = inorder.index(root_val)          # split point in inorder
        root.left  = self.buildTree(preorder[1:mid+1],  inorder[:mid])
        root.right = self.buildTree(preorder[mid+1:],   inorder[mid+1:])
        return root

preorder = [3, 9, 20, 15, 7]
inorder  = [9, 3, 15, 20, 7]
root = Solution().buildTree(preorder, inorder)
print(root.val, root.left.val, root.right.val)  # 3, 9, 20

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n^2) with list slicing + index search | O(n) – recursion and slices |
| Optimized with hashmap: O(n) | Hashmap for O(1) index lookup |
| **Final: O(n^2) naive / O(n) optimized** | **Final: O(n)** |